In [9]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


### Feature Engineering Conclusion
*Before there were 4 features, And now there are 19 features in total including-*
| Category           | Purpose             |
| ------------------ | ------------------- |
| Time features      | Seasonality         |
| Lag features       | Memory              |
| Rolling features   | Trend & volatility  |
| Aggregate features | Indicate-Behavior   |

In [2]:
df = pd.read_csv("../data/processed/featured_data.csv", parse_dates=["date"])

In [3]:
df.dtypes

date               datetime64[ns]
store                       int64
item                        int64
sales                       int64
year                        int64
month                       int64
week                        int64
day                         int64
dayofweek                   int64
is_weekend                  int64
sales_lag_1               float64
sales_lag_7               float64
sales_lag_14              float64
sales_lag_28              float64
rolling_mean_7            float64
rolling_mean_14           float64
rolling_mean_28           float64
store_avg_sales           float64
item_avg_sales            float64
dtype: object

## Data Split - Train & Validation

In [4]:
train_df = df[df["date"] < "2017-01-01"]
val_df   = df[df["date"] >= "2017-01-01"]

In [5]:

def create_sequences(series, window=28):
    X, y = [], []
    for i in range(len(series) - window):
        X.append(series[i:i+window])
        y.append(series[i+window])
    return np.array(X), np.array(y)

train_series = train_df["sales"].values
val_series   = val_df["sales"].values

X_train_seq, y_train_seq = create_sequences(train_series, 28)
X_val_seq, y_val_seq     = create_sequences(val_series, 28)

# reshape for LSTM [samples, timesteps, features]
X_train_seq = X_train_seq.reshape(-1, 28, 1)
X_val_seq   = X_val_seq.reshape(-1, 28, 1)


In [6]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train_series.reshape(-1,1))
val_scaled   = scaler.transform(val_series.reshape(-1,1))

X_train_seq, y_train_seq = create_sequences(train_scaled, 28)
X_val_seq, y_val_seq     = create_sequences(val_scaled, 28)

X_train_seq = X_train_seq.reshape(-1,28,1)
X_val_seq   = X_val_seq.reshape(-1,28,1)

In [7]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(28,1)),
    Dropout(0.2),
    LSTM(32),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse"
)

model.summary()

C:\Users\draza\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 28, 64)              │          16,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 28, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 32)                  │          12,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 29,345 (114.63 KB)

 Trainable params: 29,345 (114.63 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
es = EarlyStopping(patience=10, restore_best_weights=True)

history = model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=50,
    batch_size=64,
    callbacks=[es],
    verbose=1
)

Epoch 1/50
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 147s 13ms/step - loss: 0.0019 - val_loss: 0.0020
Epoch 2/50
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 145s 13ms/step - loss: 0.0016 - val_loss: 0.0018
Epoch 3/50
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 146s 13ms/step - loss: 0.0015 - val_loss: 0.0018
Epoch 4/50
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 147s 13ms/step - loss: 0.0014 - val_loss: 0.0017
Epoch 5/50
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 149s 13ms/step - loss: 0.0014 - val_loss: 0.0017
Epoch 6/50
  254/11195 ━━━━━━━━━━━━━━━━━━━━ 2:18 13ms/step - loss: 0.0015

In [ ]:
val_preds_scaled = model.predict(X_val_seq)
val_preds = scaler.inverse_transform(val_preds_scaled)

y_val_actual = scaler.inverse_transform(
    y_val_seq.reshape(-1,1)
)


In [ ]:

rmse_lstm = np.sqrt(mean_squared_error(y_val_actual, val_preds))
mape_lstm = np.mean(np.abs((y_val_actual - val_preds) / y_val_actual)) * 100

print("LSTM RMSE:", rmse_lstm)
print("LSTM MAPE:", mape_lstm)


In [ ]:
plt.figure(figsize=(12,4))

plt.plot(y_val_actual[:200], label="Actual Demand")
plt.plot(val_preds[:200], label="LSTM Prediction")

plt.title("LSTM Forecast vs Actual Demand")
plt.xlabel("Time Steps")
plt.ylabel("Sales")
plt.legend()
plt.show()